# ATLAS CPU reproduction

This notebook rebuilds the published descriptive tables from the audited CSV/JSON artifacts and reruns the deterministic localization summary. It does not load a model, run an agent, or claim an official Kaggle score. When original competition inputs are unavailable, it uses the small derived tables shipped with this release and prints their provenance hashes.

In [1]:
from pathlib import Path
import csv, hashlib, json, statistics
ROOT = Path.cwd()
AUDIT = ROOT / "paper" / "data" / "final_audit"
assert (AUDIT / "summary.json").exists(), f"Run from project root: {ROOT}"
summary = json.loads((AUDIT / "summary.json").read_text(encoding="utf-8"))
print({"tasks": summary["universe"]["tasks"], "nonbaseline": summary["universe"]["observed_nonbaseline_states"], "paired": summary["nonbaseline"]["paired_target_selected"], "agree": summary["nonbaseline"]["agree_target_selected"]})

{'tasks': 129, 'nonbaseline': 774, 'paired': 642, 'agree': 611}


In [2]:
rows = list(csv.DictReader((AUDIT / "cost_by_repo.csv").open(encoding="utf-8")))
for row in rows:
    print(row["repo"], row["n_pairs"], row["reduction_ratio_of_sums"])
assert {r["repo"] for r in rows} >= {"Textualize/rich", "fastapi/fastapi"}

ALL 642 0.3955445431501058
Textualize/rich 256 0.0377014905240437
encode/httpx 5 0.010657193605683735
fastapi/fastapi 333 0.6901979876663422
psf/requests 48 0.0010957812422173197


In [3]:
h2 = json.loads((ROOT / "results" / "final_h2_seed0.json").read_text(encoding="utf-8"))
print("H2 tasks:", h2["n_tasks_evaluated"], "B0 recall@50:", h2["summary"]["B0"]["recall@50"])
assert h2["n_tasks_evaluated"] == 124
assert h2["caveats"]

H2 tasks: 124 B0 recall@50: 0.1879


In [4]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
evidence = {str(p.relative_to(ROOT)): sha256(p) for p in [AUDIT / "summary.json", AUDIT / "cost_by_repo.csv", ROOT / "results" / "final_h2_seed0.json"]}
print(json.dumps({"source_hashes": evidence, "official_score": "unknown/not assessed", "model_run": "not in this notebook"}, indent=2))

{
  "source_hashes": {
    "paper\\data\\final_audit\\summary.json": "9a7e53ea670516c1a3c051ef59ced35a484e5e308edaebddd1ba95c9f6df201f",
    "paper\\data\\final_audit\\cost_by_repo.csv": "682971c9478374d14becb3a329be11841d9afd4f363fafcedab656734936d614",
    "results\\final_h2_seed0.json": "89e16e989bffad1a2a7e01c6dfea127cbbe5a964b2d5b67faeb889467772ac30"
  },
  "official_score": "unknown/not assessed",
  "model_run": "not in this notebook"
}


The assertions above are integrity checks for the released derived evidence. A successful notebook execution means the tables are internally reproducible; it does not establish test repair, model compatibility, or competition resolution.